# Imports

In [ ]:
import os
import sys

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.patches import Patch
from tqdm.auto import tqdm

from uavloc.features.segmentation import CLASSES, load_segmenter
from uavloc.plotting.correlation import SEG_PALETTE
from uavloc.plotting.segmentation import colour_labels, error_map, fixed_broken
from uavloc.segmentation.eval import HOLDOUT_LOCATION, STRIP_LOCATION, STRIP_ROWS
from uavloc.segmentation.metrics import iou_scores
from uavloc.track import REPO_ROOT, experiment_dir, resolve_device

sys.path.insert(0, str(REPO_ROOT / "pocs" / "village-seg-baseline"))
import village_seg as vs

# Params

In [ ]:
EXP = "021-building-loss-from-scratch"
ARCHITECTURES = ["unet-resnet18", "unet-resnet34"]   # add "lraspp-mobilenetv3" once its runs finish
LOSSES = ["current", "building", "structure"]
SEED = 42                                            # training seed of the checkpoints
STEPS = 2500
SAMPLE_SEED = None                                   # None: a new window every run; an int repeats one
CENTRE_CLASS = "building"                            # centre the window on a pixel of this class; None: uniform
CROP_PX = 512
GAP_PX = 64                                          # holdout margin, as in train.ipynb
DATASET = "ua-village-annotated"
DEVICE = "auto"
PANEL_IN = 3.5

# Data

In [ ]:
device = resolve_device(DEVICE)
DERIVED = REPO_ROOT / "data" / "derived"
ARTIFACTS = experiment_dir(EXP, create=False) / "artifacts"
BASELINE_LOSS = LOSSES[0]
device

In [ ]:
sat = vs.load_satellite(DERIVED / DATASET, DERIVED)
vs.ban_holdouts(sat, HOLDOUT_LOCATION, STRIP_LOCATION, STRIP_ROWS, GAP_PX)
sampler = vs.SatelliteSampler([s for s in sat if not s.banned.all()], CROP_PX, rare_p=0.0)
[(s.loc.split("-")[0], s.date) for s in sampler.images]

In [ ]:
def random_window(sampler: vs.SatelliteSampler, rng: np.random.Generator,
                  centre_class: str | None) -> tuple[int, int, int]:
    """`(image idx, top, left)` of a training window clear of holdout pixels, centred on a `centre_class` pixel."""
    if centre_class is None:
        k = rng.integers(len(sampler.images))
        top, left = sampler.corners[k][rng.integers(len(sampler.corners[k]))]
        return int(k), int(top), int(left)
    centres = sampler.centres[CLASSES.index(centre_class)]
    k, y, x = centres[rng.integers(len(centres))]
    h, w = sampler.images[k].lab.shape
    n = sampler.crop_px
    return int(k), int(np.clip(y - n // 2, 0, h - n)), int(np.clip(x - n // 2, 0, w - n))

In [ ]:
k, top, left = random_window(sampler, np.random.default_rng(SAMPLE_SEED), CENTRE_CLASS)
image = sampler.images[k]
rgb = image.img[top:top + CROP_PX, left:left + CROP_PX]
label = image.lab[top:top + CROP_PX, left:left + CROP_PX]
{"location": image.loc, "date": image.date, "top_px": top, "left_px": left}

# Model

In [ ]:
checkpoints = {(arch, loss): ARTIFACTS / f"model-{arch}-{loss}-seed{SEED}-steps{STEPS}.pt"
               for arch in ARCHITECTURES for loss in LOSSES}
missing = [p.name for p in checkpoints.values() if not p.exists()]
if missing:
    raise FileNotFoundError(f"checkpoints not in {ARTIFACTS}: {missing}")
len(checkpoints)

# Run

In [ ]:
with torch.no_grad():
    preds = {key: vs.predict(load_segmenter(path, device, key[0]), rgb, device)
             for key, path in tqdm(checkpoints.items(), desc="predict")}

In [ ]:
scores = {key: iou_scores([p], [label]) for key, p in preds.items()}
table = pd.DataFrame({key: {"miou": s["miou"], **{f"iou_{c}": v for c, v in s["iou"].items()}}
                      for key, s in scores.items()}).T
table.round(3)

# Plots

In [ ]:
def plot_panels(rgb: np.ndarray, label: np.ndarray, panels: dict[tuple[str, str], np.ndarray],
                title: str) -> plt.Figure:
    """One row per architecture: image | label | one `(H, W, 3)` panel per loss in `panels`."""
    archs = list(dict.fromkeys(a for a, _ in panels))
    losses = list(dict.fromkeys(l for _, l in panels))
    cols = ["image", "label", *losses]
    fig, axes = plt.subplots(len(archs), len(cols), figsize=(PANEL_IN * len(cols), PANEL_IN * len(archs)),
                             squeeze=False)
    for i, arch in enumerate(archs):
        for j, (ax, panel) in enumerate(zip(axes[i], [rgb, colour_labels(label), *(panels[arch, l] for l in losses)])):
            ax.imshow(panel)
            ax.set_xticks([]), ax.set_yticks([])
            ax.set_title(cols[j] if i == 0 else "")
        axes[i, 0].set_ylabel(arch)
    fig.suptitle(title)
    fig.legend(handles=[Patch(color=c / 255, label=n) for n, c in zip(CLASSES, SEG_PALETTE)],
               loc="lower center", ncol=len(CLASSES))
    fig.tight_layout(rect=(0, 0.04, 1, 1))
    return fig

In [ ]:
plot_panels(rgb, label, {key: colour_labels(p) for key, p in preds.items()}, "predictions");

In [ ]:
plot_panels(rgb, label, {key: error_map(label, p) for key, p in preds.items()}, "errors (red)");

In [ ]:
plot_panels(rgb, label, {(a, l): fixed_broken(label, preds[a, BASELINE_LOSS], p) for (a, l), p in preds.items()
                         if l != BASELINE_LOSS},
            f"vs {BASELINE_LOSS}: fixed green, broken red, both wrong grey");